## 1: Install the current SDK

In [2]:
!pip install -q google-genai

from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

In [5]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Say hello in one sentence."
)
print(response.text)

Hello, I hope you are having a wonderful day!


We used Gemini 3.6 Flash since 2.5 Flash was deprecated for new API users as of this project.

## 2: Load them to Pandas

In [6]:
import pandas as pd

benchmark = pd.read_csv("genre_occupation_benchmark.csv")
sample = pd.read_csv("ratings_sample_subset.csv")

print(benchmark.shape, sample.shape)

(6100, 4) (8102, 10)


## Prompt 1: Insights from Benchmark ( Pattern Discovery)

In [7]:
prompt = f"""
Here is a MovieLens ratings benchmark, aggregated by genre and occupation:
{benchmark.head(50).to_csv(index=False)}

This is a sample of a larger dataset with {len(benchmark)} genre/occupation combinations total.
Identify the 3 most notable patterns in how different occupations rate different genres.
Be specific and concise — not just a restatement of the numbers.
"""

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)
print(response.text)

Based on the provided dataset (using standard MovieLens occupation encodings), here are the 3 most notable rating patterns:

### 1. **Noir/Crime Hybrids Enjoy Universal High Ratings**
`Crime|Film-Noir` consistently achieves the highest average ratings across almost all occupations—ranging from **4.26 to 4.69** across lawyers (11), self-employed individuals (16), unemployed viewers (19), and unclassified users (0). This demonstrates a strong cross-demographic consensus: atmospheric, mature crime dramas are universally well-received regardless of the viewer's professional background.

### 2. **Stark Split: Family/Children's Penalty vs. Mature Animation Praise**
There is a distinct divergence in how animated and youth-oriented content is evaluated:
* **Children's & Family combinations** (e.g., `Children's|Comedy`, `Children's|Comedy|Fantasy`) are heavily penalized across adult professions—scoring consistently low (**1.67–2.94**) among academics (1), lawyers (11), and writers (20).
* **Mat

## Prompt 2: Using row-level sample ( Actionable Business Recommendations)

In [8]:
prompt2 = f"""
Here is a random sample of individual MovieLens ratings (not aggregated):
{sample.head(50).to_csv(index=False)}

This sample represents ~8,000 individual ratings from a dataset of 1,000,209 total.
Based on this sample, what would you recommend to a streaming platform trying to
improve genre-based recommendations for different age groups? Give 2-3 concrete,
actionable recommendations, not just observations.
"""

response2 = client.models.generate_content(model="gemini-3.6-flash", contents=prompt2)
print(response2.text)

Based on the rating patterns across age groups in this dataset, here are **3 concrete, actionable recommendations** to improve genre-based recommendation algorithms for different age cohorts:

---

### 1. Implement Age-Dependent Sub-Genre Weighting (Crime/Dark Drama vs. Light Comedy)
* **Observation:** The **18–25 age group** shows significantly higher engagement and top-tier ratings ($5/5$) for dark, complex, or high-intensity genres (Crime/Drama like *Pulp Fiction*, *Donnie Brasco*, *The Godfather*, and Dark Thrillers like *Crash*, *Bram Stoker's Dracula*). In contrast, generic mainstream comedies (*Groundhog Day*, *She's the One*) frequently received mixed or low ratings ($2–3/5$) from this cohort.
* **Actionable Recommendation:**
  * **Demote generic "Comedy|Romance"** in top-rail recommendations for the 18–25 age group.
  * **Boost "Crime|Drama" and "Dark Comedy|Thriller"** as primary homepage recommendations for younger users. 
  * Treat broad top-level genres (e.g., "Comedy") as

## Prompt 3: Reflect on data quality/limitations ( Critical self-assessment of  data)

In [9]:
prompt3 = f"""
Based on the genre/occupation ratings benchmark you analyzed earlier, what are
2 limitations or biases in this dataset that someone should be cautious about
before using these patterns to make real business decisions?
"""

response3 = client.models.generate_content(model="gemini-3.6-flash", contents=prompt3)
print(response3.text)

When using a genre/occupation ratings benchmark (such as the classic MovieLens benchmark or similar demographic consumer preference datasets) to inform **real business decisions**—such as product recommendations, targeted ad spending, or content acquisition—you should be particularly cautious about the following two major limitations:

---

### 1. Confounding Variables Masked as "Occupation" (Spurious Correlation)
* **The Bias:** Occupation does not exist in a vacuum; it heavily correlates with **age, gender, income level, and geographic location**. In benchmark datasets, a preference attributed to an "occupation" is frequently driven by underlying demographics that the raw correlation obscures.
  * *Example:* If the dataset shows that "Software Engineers" heavily prefer Sci-Fi or Action content, this may actually be driven by the fact that the sampled engineering demographic skewed heavily **young and male**, rather than something inherent to the profession itself.
* **Business Risk:*

In [10]:
import json

with open("gemini_analysis_results.txt", "w") as f:
    f.write("=== Prompt 1: Genre/Occupation Patterns ===\n")
    f.write(response.text + "\n\n")
    f.write("=== Prompt 2: Recommendations for Streaming Platform ===\n")
    f.write(response2.text + "\n\n")
    f.write("=== Prompt 3: Data Limitations & Biases ===\n")
    f.write(response3.text + "\n")

print("Saved gemini_analysis_results.txt")

Saved gemini_analysis_results.txt
